# Rossmann Sales Forecasting — REST API Client

Демонстрація взаємодії Jupyter Notebook із FastAPI для прогнозування щоденних продажів магазинів Rossmann.

**Мета:** перевірити доступність сервера, отримати інформацію про дані та моделі, виконати прогнозування й дослідити сценарії продажів.

## 1. Підключення до FastAPI

Використовуємо бібліотеку `requests` для надсилання HTTP-запитів та отримання JSON-відповідей.

In [1]:
import requests
from pprint import pprint

API_URL = "http://127.0.0.1:8000"
TIMEOUT = 15

print("API URL:", API_URL)

API URL: http://127.0.0.1:8000


## 2. Перевірка стану сервера

`GET /health` перевіряє доступність FastAPI.

In [2]:
response = requests.get(f"{API_URL}/health", timeout=TIMEOUT)
response.raise_for_status()
pprint(response.json())

{'service': 'Rossmann Sales Forecasting API', 'status': 'ok'}


## 3. Інформація про датасет

`GET /data/info` повертає інформацію про дані, які використовує система.

In [3]:
response = requests.get(f"{API_URL}/data/info", timeout=TIMEOUT)
response.raise_for_status()
pprint(response.json())

{'column_types': {'Assortment': 'object',
                  'CompetitionDistance': 'float64',
                  'CompetitionOpenSinceMonth': 'float64',
                  'CompetitionOpenSinceYear': 'float64',
                  'Customers': 'int64',
                  'Date': 'datetime64[ns]',
                  'DayOfWeek': 'int64',
                  'Open': 'int64',
                  'Promo': 'int64',
                  'Promo2': 'int64',
                  'Promo2SinceWeek': 'float64',
                  'Promo2SinceYear': 'float64',
                  'PromoInterval': 'object',
                  'Sales': 'int64',
                  'SchoolHoliday': 'int64',
                  'StateHoliday': 'object',
                  'Store': 'int64',
                  'StoreType': 'object'},
 'test': {'columns': 18,
          'date_max': '2015-07-31',
          'date_min': '2015-01-01',
          'rows': 21200,
          'stores': 100},
 'train': {'columns': 18,
           'date_max': '2014-12-31',
     

## 4. Отримання списку моделей

`GET /models` повертає збережені моделі та їхні метрики Validation.

In [4]:
response = requests.get(f"{API_URL}/models", timeout=TIMEOUT)
response.raise_for_status()

models = response.json()["models"]

print("Кількість моделей:", len(models))

for model in models:
    print(model["name"], "—", model["type"])

Кількість моделей: 15
keras_mlp — Keras MLP
keras_mlp_custom — Keras MLP
lightgbm — LGBMRegressor
lightgbm_custom — LightGBM
random_forest — RandomForestRegressor
random_forest_20261013_173513_dbb7aa — Random Forest
random_forest_20261013_173921_7ca657 — Random Forest
random_forest_20261013_174048_46d9f0 — Random Forest
random_forest_20261013_174213_dbc304 — Random Forest
random_forest_20261013_174443_f6eb86 — Random Forest
random_forest_20261013_192403_97f26e — Random Forest
random_forest_20261013_192533_6002e4 — Random Forest
random_forest_20261014_121345_70c8b3 — Random Forest
random_forest_20261014_123903_d677f6 — Random Forest
random_forest_custom — Random Forest


## 5. Прогнозування одного магазину

`POST /predict` повертає прогноз продажів для конкретного магазину та дати.

In [5]:
single_request = {
    "model_name": "lightgbm",
    "store": 1,
    "date": "2015-07-01",
    "promo": 1,
    "school_holiday": 0,
    "state_holiday": "0",
    "open": 1
}

response = requests.post(
    f"{API_URL}/predict",
    json=single_request,
    timeout=TIMEOUT
)
response.raise_for_status()
pprint(response.json())

{'date': '2015-07-01',
 'model': 'lightgbm',
 'predicted_sales': 5419.12,
 'store': 1}


## 6. Пакетне прогнозування

`POST /predict/batch` дозволяє отримати прогнози для кількох магазинів одним запитом.

In [6]:
batch_request = {
    "predictions": [
        {**single_request, "store": 1},
        {**single_request, "store": 2},
        {**single_request, "store": 3}
    ]
}

response = requests.post(
    f"{API_URL}/predict/batch",
    json=batch_request,
    timeout=TIMEOUT
)
response.raise_for_status()
pprint(response.json())

{'predictions': [{'date': '2015-07-01',
                  'model': 'lightgbm',
                  'predicted_sales': 5419.12,
                  'store': 1},
                 {'date': '2015-07-01',
                  'model': 'lightgbm',
                  'predicted_sales': 7264.61,
                  'store': 2},
                 {'date': '2015-07-01',
                  'model': 'lightgbm',
                  'predicted_sales': 8281.65,
                  'store': 3}],
 'total': 3}


## 7. Прогнозування на період

`POST /forecast` формує прогноз продажів для вибраних магазинів у заданому діапазоні дат.

In [7]:
forecast_request = {
    "model_name": "lightgbm",
    "stores": [1, 2, 3],
    "start_date": "2015-07-01",
    "end_date": "2015-07-07"
}

response = requests.post(
    f"{API_URL}/forecast",
    json=forecast_request,
    timeout=60
)
response.raise_for_status()
forecast_result = response.json()
pprint(forecast_result, depth=2)

{'forecasts': [{...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...}],
 'metrics': {'mae': 390.16, 'r2': 0.9321, 'rmse': 604.59, 'wape': 8.2},
 'model': 'lightgbm',
 'total': 21}


## 8. Сценарний аналіз

`POST /forecast/scenario` дозволяє дослідити, як змінюється прогноз продажів за різних умов.

Порівнюємо сценарії без акції та з акцією.

In [8]:
scenario_results = {}

for promo_value in [0, 1]:
    scenario_request = {
        **forecast_request,
        "promo": promo_value,
        "school_holiday": 0,
        "state_holiday": "0",
        "open": 1
    }

    response = requests.post(
        f"{API_URL}/forecast/scenario",
        json=scenario_request,
        timeout=60
    )
    response.raise_for_status()
    scenario_results[promo_value] = response.json()

for promo_value, result in scenario_results.items():
    print(f"\nСценарій Promo = {promo_value}")
    pprint(result, depth=2)


Сценарій Promo = 0
{'forecasts': [{...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...}],
 'mode': 'scenario',
 'model': 'lightgbm',
 'total': 21}

Сценарій Promo = 1
{'forecasts': [{...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},
               {...},


## 9. Порівняння сценаріїв продажів

Порівнюємо прогнози LightGBM для однакових магазинів і дат за двох сценаріїв: без акції (`Promo = 0`) та з акцією (`Promo = 1`).

In [9]:
import pandas as pd
import plotly.express as px

without_promo = pd.DataFrame(
    scenario_results[0]["forecasts"]
).rename(columns={"predicted_sales": "Без акції"})

with_promo = pd.DataFrame(
    scenario_results[1]["forecasts"]
).rename(columns={"predicted_sales": "З акцією"})

comparison = without_promo.merge(
    with_promo,
    on=["store", "date"],
    validate="one_to_one"
)

comparison["Різниця"] = (
    comparison["З акцією"] - comparison["Без акції"]
)

comparison["Зміна (%)"] = (
    comparison["Різниця"]
    / comparison["Без акції"].replace(0, float("nan"))
    * 100
)

display(comparison.round(2))


,store,date,Без акції,З акцією,Різниця,Зміна (%)
0,1,2015-07-01,4154.35,5419.12,1264.77,30.44
1,1,2015-07-02,4337.27,5281.43,944.16,21.77
2,1,2015-07-03,4009.21,4882.68,873.47,21.79
3,1,2015-07-04,4620.07,5789.36,1169.29,25.31
4,1,2015-07-05,5752.65,6427.64,674.99,11.73
5,1,2015-07-06,3825.85,5553.22,1727.37,45.15
6,1,2015-07-07,3971.67,5370.73,1399.06,35.23
7,2,2015-07-01,5259.69,7264.61,2004.92,38.12
8,2,2015-07-02,4347.88,5937.85,1589.97,36.57
9,2,2015-07-03,4002.60,5193.71,1191.11,29.76


In [11]:
summary = comparison.groupby(
    "store", as_index=False
)[["Без акції", "З акцією"]].sum()

chart_data = summary.melt(
    id_vars="store",
    var_name="Сценарій",
    value_name="Прогнозовані продажі"
)

fig = px.bar(
    chart_data,
    x="store",
    y="Прогнозовані продажі",
    color="Сценарій",
    barmode="group",
    title="Порівняння прогнозованих продажів за сценаріями",
    template="plotly_dark"
)

fig.update_layout(
    xaxis_title="Магазин",
    yaxis_title="Сума прогнозованих продажів",
    height=450
)

fig.show(renderer="browser")

print("Загальний прогноз без акції:",
      round(comparison["Без акції"].sum(), 2))
print("Загальний прогноз з акцією:",
      round(comparison["З акцією"].sum(), 2))


Загальний прогноз без акції: 103694.02
Загальний прогноз з акцією: 140280.46
